# Chapter 4 (part 3) — Interpreting the Regression Equation

Covers: Correlated Predictors → Multicollinearity → Confounding Variables → Interactions and Main Effects, on Ames Housing, plus a short supplementary section on mediators.

The thread: **a regression coefficient is a conditional statement, "the difference in y per unit of x among houses that look the same on everything else in the model", so what's *in* the model changes what every coefficient means.** Correlated predictors leave few houses that differ on one but not the other. Confounding means a missing variable is silently doing the work. Interactions mean "what's a square foot worth?" depends on who you ask. This notebook leans toward causal-inference questions.

**How to work each section** (~10 min each; ~1 hour total with the exam): write your one-line prediction in the *Prediction* cell **before** running anything, do the code, then write 2-3 bullets on where the result matched or broke your prediction and why. Ask for a hint if you get stuck.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.anova import anova_lm

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape

## 1. Correlated Predictors and Multicollinearity

Two predictors that move together both "explain" the same variation in y; once one is in the model, the other only gets credit for what's *left over*. Numeric picture: bigger houses have more bedrooms and cost more. Alone, a bedroom appears to add a lot of value, because it's standing in for square footage. Hold size fixed and a 4-bedroom house has smaller rooms than a 3-bedroom one.

The **variance inflation factor (VIF)** measures how much of predictor j the other predictors already explain: regress j on the others, take that R², and compute `VIF = 1 / (1 - R²)`. If the others explain 90% of j, VIF = 1/(1-0.9) = 10: its coefficient's variance is ten times what it would be with uncorrelated predictors (standard error ~3.2× bigger).

**Predict first:** when you add `Gr_Liv_Area`, will the bedroom coefficient shrink, vanish, or flip sign? Of `Gr_Liv_Area, First_Flr_SF, Total_Bsmt_SF, Year_Built, Garage_Area`, which two will have the highest VIF?

**Question 1.** (a) Fit `Sale_Price ~ Bedroom_AbvGr`, then `Sale_Price ~ Bedroom_AbvGr + Gr_Liv_Area`. Explain the change using the mean `Gr_Liv_Area` by bedroom count. What comparison does the second coefficient describe? (b) For the five predictors above, compute the VIF of `Gr_Liv_Area` by hand (auxiliary regression), then all five with `variance_inflation_factor`. Which are inflated, and why those?

*Tools for this question:*
- `df.groupby("Bedroom_AbvGr")["Gr_Liv_Area"].mean()`; read one coefficient with `res.params["Bedroom_AbvGr"]` and `res.bse["Bedroom_AbvGr"]`.
- `variance_inflation_factor(X.values, i)` (from `statsmodels.stats.outliers_influence`): `X` should be the predictor DataFrame **with** `sm.add_constant(...)`, and `i` is the column position to score.
- By hand: regress predictor j on the others, `1 / (1 - r2)`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 2. Confounding and the Omitted-Variable-Bias Identity

A **confounder** drives both the variable you care about and the outcome. Numeric picture: houses with central air sell for $60k more on average. But AC is much more common in newer houses, and old houses are cheap for many reasons unrelated to AC. Compare only houses of similar age and the AC premium shrinks.

*Supplementary (not in the book):* there's an exact accounting rule behind this. Long model `y ~ b*x + c*z`, short model `y ~ b'*x`. Then **b' = b + c × δ**, where δ is the slope from regressing z on x. Numeric picture: AC is worth $13k in the long model, each year newer is worth $1,000, and AC houses are 25 years newer on average (δ = 25). The short model's AC coefficient is 13,000 + 1,000 × 25 = $38,000. This is an identity of least squares, not an approximation.

**Predict first:** as you add `Year_Built` and then `Gr_Liv_Area`, will the `Central_Air` coefficient shrink or grow?

**Question 2.** (a) Track the `Central_Air` coefficient and standard error across (i) `Central_Air` only, (ii) `+ Year_Built`, (iii) `+ Gr_Liv_Area`. (b) Models (i) and (ii) are exactly the short and long models with z = `Year_Built`. Fit the one auxiliary regression `Year_Built ~ Central_Air` and verify the identity to the dollar. (c) Is the model-(iii) coefficient "the value of installing AC"? Name one confounder you can't observe in this dataset and the direction it would bias the estimate.

*Tools for this question:*
- 0/1 treatment column: `(df["Central_Air"] == "Y").astype(int)`. With `C(Central_Air)` instead, the coefficient is named `C(Central_Air)[T.Y]`.
- Collect results with `res.params[name]`, `res.bse[name]` in a list, then `pd.DataFrame`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 3. Interactions and Main Effects

An **interaction** lets one variable's effect depend on another's. Numeric picture: if price rises $50 per sqft for homes without central air and $110 per sqft with it, "the effect of a square foot" isn't one number. The model `y ~ x * g` includes `x`, `g`, and the product `x:g`. The coefficient on `x` is now the slope **for the reference group of g only**, the coefficient on `g` is the gap **at x = 0**, and the product term is the *difference in slope* between groups.

**Predict first:** will the price-per-sqft slope be steeper for AC or no-AC homes? Will the `Central_Air[T.Y]` coefficient itself be positive or negative?

**Question 3.** (a) Fit `Sale_Price ~ Gr_Liv_Area * C(Central_Air)` and write the fitted line for each group as an equation with an explicit slope and intercept. (b) Explain the sign of the `Central_Air[T.Y]` coefficient in terms of what `Gr_Liv_Area = 0` means. What would centering `Gr_Liv_Area` do to it? (No need to refit.) (c) How many no-AC homes are there, and how precisely is their slope estimated?

*Tools for this question:*
- `"y ~ x * C(g)"` expands to `x + C(g) + x:C(g)`. The product coefficient is named like `Gr_Liv_Area:C(Central_Air)[T.Y]`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 4. Supplementary: Controls That Hurt (Mediators)

*(Beyond the book, but the natural follow-up to confounding.)* If a variable sits *between* the cause and the effect (a **mediator**), controlling for it blocks the pathway you're trying to measure. Numeric picture: a remodel works partly by raising overall quality, which buyers pay for. Control for quality and you're comparing remodeled and unremodeled homes *of the same quality*, which wipes out the main way remodeling helps.

Data quirk: `Year_Remod_Add` is floored at 1950, so for older homes `Year_Remod_Add > Year_Built` is true whether or not anything was remodeled. Restrict to `Year_Built >= 1950` and define `remod = (Year_Remod_Add > Year_Built)`.

**Predict first:** when you add overall quality to the model, will the `remod` coefficient go up, go down, or stay about the same?

**Question 4.** (a) Fit `Sale_Price ~ remod`, then `+ Year_Built + Gr_Liv_Area`, then `+ numeric overall quality` (reuse your 1-10 mapping from the factor-variables notebook). Report the `remod` coefficient each time. (b) Which one would you report as "the effect of remodeling on price", and why is the last one the wrong answer to that question?

*Tools for this question:*
- `(df["Year_Remod_Add"] > df["Year_Built"]).astype(int)` for the flag; `df.query("Year_Built >= 1950")` to restrict.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

---
# Exam — check your intuition

5 questions, about 3-4 minutes each. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, 2-3 bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Two correlated predictors in one model
*Role lean: DS · ML*

Garage capacity can be measured by number of cars or by square feet — two measures of nearly the same thing. Three models:

```text
corr(Garage_Cars, Garage_Area) = 0.89      VIF of each predictor in the two-variable model = 4.8

MODEL A: Sale_Price~Garage_Cars
                coef  std err     t   P>|t|
Intercept    60753.4   2842.6  21.4  <0.001
Garage_Cars  67966.2   1478.0  46.0  <0.001
   held-out RMSE (70/30 split) = 63,308

MODEL B: Sale_Price~Garage_Area
                coef  std err     t   P>|t|
Intercept    68470.4   2737.3  25.0  <0.001
Garage_Area    237.6      5.3  45.1  <0.001
   held-out RMSE (70/30 split) = 63,020

MODEL C: Sale_Price~Garage_Cars+Garage_Area
                coef  std err     t   P>|t|
Intercept    57526.7   2812.8  20.5  <0.001
Garage_Cars  39294.5   3185.1  12.3  <0.001
Garage_Area    114.0     11.3  10.1  <0.001
   held-out RMSE (70/30 split) = 61,932
```

(a) Both variables are strongly "significant" in Model C, yet the coefficients are about half of what they were alone. Say in plain words what the `Garage_Cars` coefficient of about 39,300 in Model C means versus the roughly 68,000 in Model A. (b) The standard error on `Garage_Area` more than doubled from Model B to Model C. Why? (c) Held-out RMSE barely moved across the three models. What does that tell you about when multicollinearity is and isn't a problem — for *prediction* versus *explaining which variable matters*? (d) A colleague says "VIF is only about 5, so there's no problem." What would you want to know before agreeing?

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — Why the coefficients wobble but the prediction doesn't
*Role lean: DS · ML · interview*

Continuing with Model C, here is how much its numbers jump around if you re-fit on bootstrap resamples of the data:

```text
1,000 bootstrap re-fits of MODEL C (Sale_Price ~ Garage_Cars + Garage_Area):
  std dev of the Garage_Cars coefficient across re-fits : 4,056  (full-data estimate 39,295)
  std dev of the Garage_Area coefficient across re-fits : 15.9  (full-data estimate 114.0)
  correlation between the two coefficient estimates     : -0.87
  predicted price for a 2-car, 500 sq ft garage: mean 193,153, std dev across re-fits 1,397
```

(a) The two coefficient estimates are strongly *negatively* correlated across re-fits. Explain intuitively why — what has to happen to the Area coefficient when the Cars coefficient bounces up? (b) Why is the *predicted price* so much more stable than either coefficient? (c) What does this say about whether you'd report "the effect of one more car stall" from this model to a stakeholder? What would you do instead (name two options)?

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Predicting the sign of omitted-variable bias
*Role lean: Causal (supplementary) · DS · interview*

A short model leaves out `Year_Built`; a long model includes it. Everything below is from real fits:

```text
SHORT model: Sale_Price ~ Garage_Area
   Garage_Area coef = 237.6

LONG model:  Sale_Price ~ Garage_Area + Year_Built
   Garage_Area coef = 179.5     Year_Built coef = 861.2

Auxiliary:   Year_Built ~ Garage_Area
   Garage_Area coef = 0.0676   (each extra sq ft of garage goes with a house that is 0.0676 years newer, on average)
```

(a) Before looking at the numbers, which direction would you have predicted the bias to go (does the short-model coefficient on Garage_Area overstate or understate the long-model one), using only: "newer houses sell for more" and "newer houses have bigger garages"? (b) Verify the identity short = long + (Year_Built coef in long) × (Garage_Area coef in auxiliary) with the numbers above. (c) State the identity in words and use it to answer: if the omitted variable were *uncorrelated* with the one you kept, what would happen? If it were *negatively* correlated with the kept variable but positively related to price, which way would the bias go? (d) Why is this the formula behind most "controlling for X" conversations in causal inference?

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — A control that hurts
*Role lean: Causal (supplementary) · DS · interview*

An agent wants to know what *adding a full bathroom* is worth. Here is the coefficient on `Full_Bath` as controls are added:

```text
Coefficient on Full_Bath (std err), by what else is in the model:
  Full_Bath only                         78,827  (2,238)
  + Year_Built                           52,530  (2,313)
  + Year_Built + Gr_Liv_Area             -8,856  (2,208)

Auxiliary:  Gr_Liv_Area ~ Full_Bath + Year_Built  ->  coefficient on Full_Bath = 606 sq ft per extra full bath
Full_Bath counts: 0 baths: 12, 1 baths: 1318, 2 baths: 1532, 3 baths: 64, 4 baths: 4
```

(a) Walk through the three coefficients: why does adding `Year_Built` shrink the number, and why does the last step flip the sign? (b) The auxiliary regression says each extra full bath goes with about 606 more square feet of living area. In causal-inference language, what role does `Gr_Liv_Area` play if the question is "what is a bathroom *addition* worth" — and what question does the third coefficient actually answer instead? (c) Which of the three numbers would you report for the agent's question, and which for a question like "among homes of identical size, does swapping a bedroom for a second bath raise price"? (d) Interview version: "You're told to include every available covariate to be safe. What could go wrong?" Give an example from this table.

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — Reading an interaction
*Role lean: DS · Causal · interview*

A model that lets the price-per-square-foot differ between single-family homes and end-unit townhouses:

```text
Sale_Price ~ Gr_Liv_Area * C(Bldg_Type), only OneFam and TwnhsE homes
(reference level = OneFam)
                                       coef  std err     t   P>|t|
Intercept                            6094.2   3419.2   1.8   0.075
C(Bldg_Type)[T.TwnhsE]             -27118.1  17321.1  -1.6   0.118
Gr_Liv_Area                           117.7      2.1  55.3  <0.001
Gr_Liv_Area:C(Bldg_Type)[T.TwnhsE]     41.2     12.5   3.3   0.001

n = 2658   R-squared = 0.549   Adj. R-squared = 0.548   Residual std error = 54,801
Average Gr_Liv_Area: OneFam 1,518, TwnhsE 1,342
```

(a) What is the estimated price per extra sq ft for a OneFam home? For a TwnhsE home? (b) The `C(Bldg_Type)[T.TwnhsE]` coefficient is -27,118. Does that mean townhouses sell for $27k less? Where does that number apply, and is that a place you have data? (c) Compute the predicted TwnhsE-minus-OneFam price gap at 1,000 sq ft and at 2,500 sq ft. At roughly what size do the two fitted lines cross? (d) How would centering `Gr_Liv_Area` at its mean change the dummy coefficient's meaning, and what would stay the same? (e) The interaction has p = 0.001 but the TwnhsE main effect has p = 0.118. A teammate says "drop the non-significant main effect." Do you agree?

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_